In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [4]:
class CNNLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        # =========================
        # CNN
        # =========================

        # Input : (B, 1, 30)
        # Output : (B, 256, 29)
        self.conv1 = nn.Conv1d(
            in_channels=1,
            out_channels=256,
            kernel_size=2
        )

        # (B, 256, 29)
        # -> (B, 128, 28)
        self.conv2 = nn.Conv1d(
            in_channels=256,
            out_channels=128,
            kernel_size=2
        )

        # (B, 128, 28)
        # -> (B, 128, 14)
        self.pool = nn.MaxPool1d(
            kernel_size=2
        )

        # =========================
        # LSTM
        # =========================

        # Après Flatten :
        # 128 * 14 = 1792

        # RepeatVector(30)
        self.lstm1 = nn.LSTM(
            input_size=1792,
            hidden_size=100,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.2)

        self.lstm2 = nn.LSTM(
            input_size=100,
            hidden_size=100,
            batch_first=True
        )

        # Bidirectional LSTM
        self.bilstm = nn.LSTM(
            input_size=100,
            hidden_size=100,
            batch_first=True,
            bidirectional=True
        )

        # =========================
        # Dense
        # =========================

        # Bidirectional -> 100 * 2 = 200
        self.fc1 = nn.Linear(200, 100)

        self.fc2 = nn.Linear(100, 1)

    def forward(self, x):

        # x initialement :
        # (B, 30, 1)

        # Conv1d attend :
        # (B, channels, length)

        x = x.transpose(1, 2)
        # (B, 1, 30)

        # CNN
        x = self.conv1(x)
        # (B, 256, 29)

        x = self.conv2(x)
        # (B, 128, 28)

        x = self.pool(x)
        # (B, 128, 14)

        # Flatten
        x = x.flatten(start_dim=1)
        # (B, 1792)

        # RepeatVector(30)
        x = x.unsqueeze(1)
        # (B, 1, 1792)

        x = x.repeat(1, 30, 1)
        # (B, 30, 1792)

        # LSTM 1
        x, _ = self.lstm1(x)
        # (B, 30, 100)

        # Dropout
        x = self.dropout(x)

        # LSTM 2
        x, _ = self.lstm2(x)
        # (B, 30, 100)

        # Bidirectional LSTM
        x, _ = self.bilstm(x)
        # (B, 30, 200)

        # Dense
        x = self.fc1(x)
        # (B, 30, 100)

        x = self.fc2(x)
        # (B, 30, 1)

        return x